# Answering from fact Knowledge Indicators

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elastic/elasticsearch-labs/blob/main/supporting-blog-content/precomputed-context-technical-walkthrough-part-2/index-facts-kis.ipynb)

This notebook is the runnable companion to [Know Your Facts: Context Management for Smarter AI Agents, Powered by Elasticsearch AI Indices](https://www.elastic.co/search-labs/blog/context-management-technical-walkthrough-facts), Part 2 of the Precomputed context technical walkthrough series.

[Part 1](https://www.elastic.co/search-labs/blog/context-management-technical-walkthrough-index-metadata) showed how routing tells an agent *where* to look. In Part 2, we go one level deeper: we pre-compute the actual facts so an agent can retrieve an answer without reading a full document. We generate one `corpus_entry` Knowledge Indicator (KI) per document with a Kibana Workflow — here scoped to a curated handful of documents the example question depends on, so the run stays fast and cheap — store them in an AI Index, and compare an agent answering the same question with and without those KIs.

## Prerequisites

This notebook is designed to run against an Elastic Serverless project, where the `ai-index-` component templates and Kibana Workflows are available. If you don't have one, [sign up for a trial](https://cloud.elastic.co/registration?onboarding_token=search&cta=cloud-registration&tech=trial&plcmt=article%20content&pg=search-labs).

Before you start:

- Have your Elasticsearch and Kibana endpoint URLs and an API key ready.
- Configure a GenAI connector in Kibana (Stack Management → Connectors) for the workflow's `ai.agent` step. Serverless projects come pre-configured with the Elastic Inference Service.
- Provide an LLM API key for any OpenAI-compatible endpoint, used by the deep agent harness. It defaults to OpenRouter with Claude Sonnet 4.5.

## Install packages and import modules

In [ ]:
!pip install -q "elasticsearch>=9,<10" datasets requests langchain-openai langchain-core deepagents

### Initialize the Elasticsearch client

Connect with your Elasticsearch and Kibana endpoint URLs and an API key. The same inputs work for an Elastic Cloud deployment or a Serverless project. If you don't have an API key, you can [create one using these instructions](https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key).

In [ ]:
import json
import time
import requests
from getpass import getpass
from elasticsearch import Elasticsearch, helpers

ES_URL = input("Elasticsearch endpoint URL: ").strip().rstrip("/")
KIBANA_URL = input("Kibana endpoint URL: ").strip().rstrip("/")
ELASTIC_API_KEY = getpass("Elastic API key: ")

client = Elasticsearch(hosts=[ES_URL], api_key=ELASTIC_API_KEY)
print(client.info())

Confirm Kibana is reachable. We drive the Workflows API through it.

In [ ]:
def kbn_request(method, path, *, body=None, api_version=None):
    """Call a Kibana REST API and return the parsed JSON response."""
    headers = {
        "Authorization": f"ApiKey {ELASTIC_API_KEY}",
        "kbn-xsrf": "true",
        "Content-Type": "application/json",
    }
    if api_version:
        headers["elastic-api-version"] = api_version
    resp = requests.request(
        method,
        f"{KIBANA_URL}{path}",
        headers=headers,
        data=json.dumps(body) if body is not None else None,
    )
    resp.raise_for_status()
    return resp.json() if resp.text else {}


status = kbn_request("GET", "/api/status")
print("Kibana status:", status.get("status", {}).get("overall", {}).get("level"))

## Create some sample data

We use a small sample of the [BrowseComp-Plus](https://github.com/texttron/BrowseComp-Plus) corpus, a reasoning-intensive browsing/QA retrieval benchmark, indexed BM25-only with `docid`, `url`, `title`, and `text` fields plus mapping metadata. We derive a title from each document's front matter. This raw corpus is the baseline haystack; later, to keep the walkthrough fast and focused, the workflow generates KIs for only a curated handful of these documents (the ones the example question depends on), not the whole sample.

In [ ]:
import re

INDEX_NAME = "browsecomp-plus"
SAMPLE_DOCS = 50  # raw corpus = baseline haystack; the workflow builds KIs for only the curated KI_DOCIDS, which must fall within this sample

MAPPINGS = {
    "_meta": {
        "description": (
            "BrowseComp-Plus corpus: ~100k human-verified web documents "
            "(news articles, Wikipedia entries, institutional pages) used as a "
            "reasoning-intensive browsing/QA retrieval benchmark. BM25-only index."
        )
    },
    "properties": {
        "docid": {
            "type": "keyword",
            "meta": {"description": "Stable corpus document id."},
        },
        "url": {
            "type": "keyword",
            "meta": {"description": "Source URL the document was crawled from."},
        },
        "title": {
            "type": "text",
            "meta": {
                "description": "Document title (from the document's front matter)."
            },
        },
        "text": {
            "type": "text",
            "meta": {
                "description": "Full document text: title, date, and body content."
            },
        },
    },
}


def extract_title(text):
    # Each BrowseComp-Plus doc opens with a YAML front-matter block whose `title:`
    # field holds the real title; fall back to the first non-delimiter line.
    m = re.search(r"^title:\s*(.+)$", text, flags=re.MULTILINE)
    if m:
        return m.group(1).strip()[:200]
    for line in text.splitlines():
        s = line.strip()
        if s and s != "---":
            return s[:200]
    return ""


if (
    client.indices.exists(index=INDEX_NAME)
    and client.count(index=INDEX_NAME)["count"] > 0
):
    count = client.count(index=INDEX_NAME)["count"]
    print(
        f"Index '{INDEX_NAME}' already has {count} documents -- reusing it (skipping indexing)."
    )
else:
    client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
    client.indices.create(index=INDEX_NAME, mappings=MAPPINGS)

    from datasets import load_dataset

    corpus = load_dataset(
        "Tevatron/browsecomp-plus-corpus", split="train", streaming=True
    )

    def actions(n):
        for i, row in enumerate(corpus):
            if i >= n:
                break
            text = row["text"]
            yield {
                "_index": INDEX_NAME,
                "_id": row["docid"],
                "_source": {
                    "docid": row["docid"],
                    "url": row["url"],
                    "title": extract_title(text),
                    "text": text,
                },
            }

    helpers.bulk(client, actions(SAMPLE_DOCS))
    client.indices.refresh(index=INDEX_NAME)
    print(
        f"Indexed {client.count(index=INDEX_NAME)['count']} documents into '{INDEX_NAME}'."
    )

## Create your AI Index

KIs live in an AI Index. The naming convention is what triggers automatic configuration: any index whose name starts with `ai-index-idx-` is a standard index, and `ai-index-ds-` is a data stream. When Elasticsearch sees the prefix, it applies component templates that configure the right mappings. The `title`, `description`, and `content` fields each get a `.semantic` [`semantic_text`](https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/semantic-text) sub-field for hybrid retrieval, alongside `type`, `tags`, `attributes`, and `references`. We will use a standard index, so we start the name with `ai-index-idx-`.

In [ ]:
AI_INDEX = "ai-index-idx-my-corpus"

# Recreate the AI Index. Passing no mappings lets the `ai-index-` component
# templates configure the standard KI fields (title/description/content with
# semantic_text sub-fields, plus type, tags, attributes, references).
client.indices.delete(index=AI_INDEX, ignore_unavailable=True)
client.indices.create(index=AI_INDEX)
print(f"Created AI Index: {AI_INDEX}\n")

print(json.dumps(client.indices.get_mapping(index=AI_INDEX).body, indent=2))

## The query-ki skill

A KI is just a document in the AI Index, and finding one is a single ES|QL query. We package that query as a small, portable [Agent Skill](https://www.anthropic.com/news/skills): a `SKILL.md` with a YAML header plus instructions, so any harness can load it. The query runs a hybrid (lexical and semantic) search, fuses the results with RRF, and filters by KI `type` (here, `corpus_entry`).

We write it to disk so the deep agent below can load it from the `skills/` directory.

In [ ]:
import os

SKILL_MD = """---
name: query-ki
description: >-
  Retrieve Knowledge Indicators (pre-computed context) from the Elasticsearch AI
  Index before answering. Use it to find which index to search (routing profiles)
  or to look up pre-computed facts without reading source documents. Trigger on any question that depends on specific facts, names, dates, or on choosing a data source.
allowed-tools: esql_query
---

# Retrieving Knowledge Indicators

Knowledge Indicators (KIs) live in Elasticsearch indices named `ai-index-*`.
Retrieve them by calling the `esql_query` tool with the query below. Substitute
the user's question for `<query>`, and choose the KI type you need: `corpus_entry`
for facts, `index_metadata_entry` for routing profiles.

```esql
FROM ai-index-idx-* METADATA _id, _index, _score
| WHERE type == "<ki_type>"
| FORK
    (WHERE MATCH(content, "<query>") OR MATCH(description, "<query>")
     | SORT _score DESC | LIMIT 20)
    (WHERE MATCH(content.semantic, "<query>") OR MATCH(description.semantic, "<query>")
     | SORT _score DESC | LIMIT 20)
| FUSE
| SORT _score DESC
| KEEP title, content, description, tags
| LIMIT 5
```

Ground your answer in what the query returns, and cite the KI titles you used. If
nothing relevant comes back, say so rather than guessing.
"""

os.makedirs("skills/query-ki", exist_ok=True)
with open("skills/query-ki/SKILL.md", "w") as f:
    f.write(SKILL_MD)
print("Wrote skills/query-ki/SKILL.md")

## Baseline: an agent without KIs

First we'll establish a baseline with an agent that has no access to KIs.

As a raw baseline, here's the hybrid RRF query the agent's `esql_query` tool would run against the raw corpus. It drops several hundred words of body text into the model's context per hit. That may work, but it's expensive, and the cost compounds with every miss.

In [ ]:
QUESTION = "What was the actress who played Torvi from Vikings also known for?"

resp = client.esql.query(
    query=f"""
        FROM {INDEX_NAME} METADATA _score, _id, _index
        | FORK
            (WHERE match(title, "{QUESTION}") | SORT _score DESC | LIMIT 100)
            (WHERE match(text,  "{QUESTION}") | SORT _score DESC | LIMIT 100)
        | FUSE
        | SORT _score DESC
        | KEEP _id, title, text
        | LIMIT 3
    """,
    format="json",
)
cols = [c["name"] for c in resp["columns"]]
for row in resp["values"]:
    r = dict(zip(cols, row))
    print(f"[{r['title']}] ({len(r['text'])} chars of body text)")

Now the baseline agent: its only tools are raw ES|QL over `browsecomp-plus` and `get_mapping`.

In [ ]:
import threading

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langchain_core.callbacks import get_usage_metadata_callback
from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend

# Any OpenAI-compatible endpoint works (OpenRouter, OpenAI, a local vLLM/Ollama, ...).
# Press Enter at the prompts to accept the defaults: OpenRouter + Claude Sonnet 4.5.
LLM_BASE_URL = (
    input("LLM base URL [https://openrouter.ai/api/v1]: ").strip()
    or "https://openrouter.ai/api/v1"
)
LLM_MODEL = (
    input("LLM model [anthropic/claude-sonnet-4.5]: ").strip()
    or "anthropic/claude-sonnet-4.5"
)
LLM_API_KEY = getpass("LLM API key: ")

# The deep agent plans and may chain several tool calls, so give it output headroom.
agent_llm = ChatOpenAI(
    base_url=LLM_BASE_URL,
    model=LLM_MODEL,
    api_key=LLM_API_KEY,
    max_tokens=4096,
)


@tool
def esql_query(query: str) -> list[dict] | str:
    """Execute an ES|QL query against Elasticsearch and return the matching rows.

    Args:
        query: A complete ES|QL query string, e.g. 'FROM browsecomp-plus | LIMIT 5'.
               Full-text search syntax: WHERE MATCH(field, "value") -- not field MATCH "value".
    """
    try:
        resp = client.esql.query(query=query, format="json")
        cols = [c["name"] for c in resp["columns"]]
        return [dict(zip(cols, row)) for row in resp["values"]]
    except Exception as e:
        return f"ES|QL error: {e}"


@tool
def get_mapping(index: str) -> dict:
    """Return the field mapping for an Elasticsearch index or pattern."""
    return client.indices.get_mapping(index=index).body


def run_agent(agent, question):
    """Invoke the agent, print the tool calls it made and its answer.

    The headline metric is the number of tool calls -- with KIs the agent reaches
    the same answer in far fewer steps. Wall-clock latency and token usage are
    printed as bonus cost signals.
    """
    print("Running agent", end="", flush=True)
    done = threading.Event()

    def _heartbeat():
        while not done.wait(5):
            print(".", end="", flush=True)

    hb = threading.Thread(target=_heartbeat, daemon=True)
    hb.start()
    start = time.time()
    try:
        with get_usage_metadata_callback() as cb:
            result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    finally:
        done.set()
        hb.join()
    elapsed = time.time() - start
    print()

    print("\n--- Tool calls ---")
    for m in result["messages"]:
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                print(f"  [{tc['name']}] {str(tc['args'])[:120]}")
    total = sum(
        len(m.tool_calls)
        for m in result["messages"]
        if isinstance(m, AIMessage) and m.tool_calls
    )
    print(f"Total: {total}\n")

    usage = next(iter(cb.usage_metadata.values()), {})
    print(f"[latency] {elapsed:.1f}s")
    print(
        f"[tokens] input={usage.get('input_tokens', 0):,}  "
        f"output={usage.get('output_tokens', 0):,}  "
        f"total={usage.get('total_tokens', 0):,}\n"
    )

    print("--- Answer ---")
    print(result["messages"][-1].content)

In [ ]:
baseline_agent = create_deep_agent(
    model=agent_llm,
    tools=[esql_query, get_mapping],
    system_prompt=(
        "You are a research assistant answering questions about a document corpus "
        "stored in the Elasticsearch index `browsecomp-plus` (fields: docid, url, "
        "title, text). You have NOT memorized the corpus. Answer by querying the raw "
        "index directly with ES|QL via the esql_query tool. "
        'Full-text search syntax: WHERE MATCH(field, "value") -- never use field MATCH "value". '
        "Use get_mapping if you are unsure of field names. Ground your answer strictly "
        "in the rows returned, and cite the docid or url you used."
    ),
)

run_agent(baseline_agent, QUESTION)

## Generate fact KIs with a Kibana Workflow

This workflow reads documents with a single ES|QL query and writes one fact-level KI per document into the AI Index. Because generating a KI is an LLM call per document, we filter that query to a curated set of document IDs (`KI_DOCIDS` below) — the specific documents this question depends on — so the workflow generates at most a handful of KIs (≤ 10) rather than one per corpus document. Each iteration runs two steps:

| Step | Type | What it does |
|------|------|--------------|
| `generate_ki` | `ai.agent` | Distill a raw document into a structured KI (title, summary, questions it answers, key entities, topics). |
| `sink_ki` | `elasticsearch.bulk` | Write the KI to the AI Index as a `corpus_entry`, keyed on `docid` so re-runs upsert in place. |

### Define the workflow

This is the similar to the YAML you'd paste into the Workflows UI in the blog, templated so the sink writes to the AI Index you created above. The only difference is that we're surgically creating KIs based on doc IDs for this example. In a real scenario, generating KIs for every document, this Workflow would process significantly more documents and take several minutes or longer to run. 

In [ ]:
# Generating a KI is an LLM call per document, so we don't build one for every
# doc in the corpus. Instead we curate the exact documents this walkthrough's
# question depends on -- the Torvi / Georgia Hirst cluster -- and generate KIs
# for only those (<= 10). KIs are keyed on docid, so this set stays stable
# across re-runs (re-runs upsert the same _id in place).
KI_DOCIDS = [
    "11589",  # Georgia Hirst - Wikipedia (holds the answer: also known for Ravers)
    "50639",  # Georgia Hirst - Vikings fandom (portrays Torvi)
    "64501",  # Torvi - the character
    "41758",  # Vikings season 6 - Wikipedia (cast list)
    "57766",  # Vikings (TV series) - Wikipedia
    "84983",  # Georgia Hirst body measurements
    "82008",  # Maude Hirst - Wikipedia (sister; near-miss distractor)
]

_WORKFLOW_YAML_TEMPLATE = """
version: '1'
name: browsecomp-plus-doc-ki
description: Query the BrowseComp-Plus corpus with ES|QL, generate a KI per doc with an AI agent, and bulk-write each into the AI Index as a corpus_entry.
enabled: true
tags:
  - precomputed-context
  - browsecomp-plus
triggers:
  - type: manual
steps:
  - name: query_corpus
    type: elasticsearch.esql.query
    with:
      # WHERE drops empty bodies and restricts to the curated KI_DOCIDS -- the
      # specific documents this example's question depends on -- so the workflow
      # generates only a handful of KIs instead of one per corpus document.
      # SUBSTRING keeps the prompt bounded (a full body would blow the context window).
      # Column order drives the foreach.item[N] indices:
      #   item[0]=docid  item[1]=title  item[2]=url  item[3]=text
      query: >
        FROM browsecomp-plus
        | WHERE text IS NOT NULL AND docid IN (__KI_DOCIDS__)
        | KEEP docid, title, url, text
        | EVAL text = SUBSTRING(text, 1, 12000)

  - name: loop_corpus_docs
    type: foreach
    foreach: '{{ steps.query_corpus.output.values }}'
    steps:
      # Turn the raw doc into a retrieval-optimized Knowledge Indicator.
      - name: generate_ki
        type: ai.agent
        timeout: 300s
        with:
          message: >
            You are a knowledge engineer building a Knowledge Indicator (KI)
            for an enterprise document-retrieval corpus. A KI is a compact,
            high-signal record that a hybrid (BM25 + semantic) search engine
            and an AI agent use to FIND and JUDGE the source document without
            reading it in full.

            Read the document below and extract a faithful, richly structured KI.
            Follow these rules strictly:
            - Be 100% grounded: never state anything not supported by the text.
            - Prefer concrete, named specifics (people, organizations, products,
              dates, places, figures) over vague phrasing.
            - Write for retrieval, not prose flourish. No marketing language.
            - If a field cannot be determined from the text, return an empty
              string or empty array rather than guessing.

            Document ID: {{ foreach.item[0] }}
            Original Title: {{ foreach.item[1] }}
            Source URL: {{ foreach.item[2] }}
            Document Body:
            {{ foreach.item[3] }}
          schema:
            type: object
            properties:
              title:
                type: string
                description: A concise, specific, human-readable title (<= 12 words).
              summary:
                type: string
                description: A dense 3-5 sentence factual summary capturing the document's main claims, named entities, and conclusions. PRIMARY semantic search surface.
              answers_questions:
                type: array
                items:
                  type: string
                description: 2-5 natural-language questions this document can authoritatively answer.
              key_entities:
                type: array
                items:
                  type: string
                description: 3-10 salient named entities (people, organizations, products, places, dates) explicitly mentioned in the text.
              topics:
                type: array
                items:
                  type: string
                description: 3-8 short topic/category labels.
              tagline:
                type: string
                description: A single ultra-short phrase (<= 6 words) as a quick-reference label.
            required:
              - title
              - summary
              - answers_questions
              - key_entities
              - topics

      # Direct bulk write to the AI Index. The explicit `index` action row sets
      # _id = docid so re-runs upsert in place (idempotent). `index:` in `with`
      # supplies the default target index for the bulk request.
      - name: sink_ki
        type: elasticsearch.bulk
        with:
          index: __AI_INDEX__
          operations:
            - index:
                _id: '{{ foreach.item[0] }}'
            - '@timestamp': '{{ execution.startedAt | date: "%Y-%m-%dT%H:%M:%S.%LZ" }}'
              type: corpus_entry
              title: '{{ foreach.item[1] | default: steps.generate_ki.output.structured_output.title }}'
              tags:
                - browsecomp-plus
              references:
                uri: '{{ foreach.item[2] }}'
              attributes:
                docid: '{{ foreach.item[0] }}'
                url: '{{ foreach.item[2] }}'
                source_index: browsecomp-plus
                tagline: '{{ steps.generate_ki.output.structured_output.tagline }}'
                topics: '{{ steps.generate_ki.output.structured_output.topics | json }}'
                answers_questions: '{{ steps.generate_ki.output.structured_output.answers_questions | json }}'
                key_entities: '{{ steps.generate_ki.output.structured_output.key_entities | json }}'
              content: >
                === SOURCE / PROVENANCE ===
                Backing Elasticsearch index: browsecomp-plus
                Document ID (docid): {{ foreach.item[0] }}
                Source URL: {{ foreach.item[2] }}
                Retrieve the full original document with ES|QL:
                FROM browsecomp-plus | WHERE docid == "{{ foreach.item[0] }}"
                === KNOWLEDGE INDICATOR ===
                {{ steps.generate_ki.output.structured_output.summary }}
                Questions this document answers: {{ steps.generate_ki.output.structured_output.answers_questions | join: " | " }}
                Key entities: {{ steps.generate_ki.output.structured_output.key_entities | join: ", " }}
              description: >
                {{ steps.generate_ki.output.structured_output.tagline }}.
                Topics: {{ steps.generate_ki.output.structured_output.topics | join: ", " }}.
                Entities: {{ steps.generate_ki.output.structured_output.key_entities | join: ", " }}.
"""

WORKFLOW_YAML = _WORKFLOW_YAML_TEMPLATE.replace("__AI_INDEX__", AI_INDEX).replace(
    "__KI_DOCIDS__", ", ".join(f'"{d}"' for d in KI_DOCIDS)
)
print(WORKFLOW_YAML)

### Create and run the workflow

The `foreach` loop runs sequentially and each iteration makes an LLM call, so this takes a few minutes for the sample. For scale, use [workflow.executeAsync](https://www.elastic.co/docs/explore-analyze/workflows/steps/composition) or native parallel support.

In [ ]:
WF_API_VERSION = "2023-10-31"
TERMINAL_STATES = {"completed", "failed", "cancelled", "timed_out", "skipped"}


def create_workflow(yaml_str):
    return kbn_request(
        "POST",
        "/api/workflows/workflow",
        body={"yaml": yaml_str},
        api_version=WF_API_VERSION,
    )


def run_workflow(workflow_id, inputs=None):
    return kbn_request(
        "POST",
        f"/api/workflows/workflow/{workflow_id}/run",
        body={"inputs": inputs or {}},
        api_version=WF_API_VERSION,
    )


def wait_for_execution(
    execution_id, timeout=1800, interval=5, total=None, iteration_step=None
):
    deadline = time.time() + timeout
    start = time.time()
    last_status = None
    last_done = -1
    dotting = False

    # Each doc KI is an LLM call, so the run goes quiet for a minute or more
    # between updates. Print a "." on every idle poll so it's clear the notebook
    # is still working, and break to a fresh line before real updates.
    def log(msg):
        nonlocal dotting
        if dotting:
            print()
            dotting = False
        print(msg)

    while time.time() < deadline:
        ex = kbn_request(
            "GET",
            f"/api/workflows/executions/{execution_id}?includeOutput=true",
            api_version=WF_API_VERSION,
        )
        status = ex["status"]
        progressed = False
        if status != last_status:
            last_status = status
            log(f"  [{int(time.time() - start):>4}s] execution: {status}")
            progressed = True
        if total and iteration_step:
            done = sum(
                1
                for s in ex.get("stepExecutions", [])
                if iteration_step in (s.get("stepId") or "")
                and s.get("status") == "completed"
            )
            done = total if status == "completed" else min(done, total)
            if done != last_done:
                last_done = done
                filled = int(30 * done / total)
                bar = "#" * filled + "." * (30 - filled)
                log(f"  [{int(time.time() - start):>4}s] {done}/{total} [{bar}]")
                progressed = True
        if status in TERMINAL_STATES:
            if dotting:
                print()
            return ex
        if not progressed:
            print(".", end="", flush=True)
            dotting = True
        time.sleep(interval)
    raise TimeoutError(f"Execution {execution_id} did not finish within {timeout}s")

In [ ]:
wf = create_workflow(WORKFLOW_YAML)
workflow_id = wf["id"]
print("Created workflow:", workflow_id)

execution = run_workflow(workflow_id)
exec_id = execution["workflowExecutionId"]
print("Running execution:", exec_id, "-- this may take a few minutes...\n")

# One KI is written per curated docid, so the progress bar tracks len(KI_DOCIDS).
result = wait_for_execution(
    exec_id,
    total=len(KI_DOCIDS),
    iteration_step="sink_ki",
)
print("Status:", result["status"])
if result["status"] != "completed":
    if result.get("error"):
        print("Error:", result["error"].get("message", result["error"]))
    for step in result.get("stepExecutions", []):
        if step.get("status") == "failed":
            print(
                "Failed step:", step.get("stepId"), "->", json.dumps(step.get("error"))
            )

### Inspect the results

Query the AI Index directly to browse the fact KIs the workflow wrote.

In [ ]:
resp = client.esql.query(
    query=f"""
        FROM {AI_INDEX}
        | WHERE type == "corpus_entry"
        | KEEP title, description, attributes, tags
        | LIMIT 25
    """,
    format="json",
)
cols = [c["name"] for c in resp["columns"]]
for row in resp["values"]:
    print(json.dumps(dict(zip(cols, row)), indent=2))

## Hand it to an agent: answering from KIs

Same harness, same question, but now the agent has the `query-ki` skill. Instead of reading raw documents, it retrieves pre-computed fact KIs and grounds its answer in them. Compare the tool-call count and the answer with the baseline above.

In [ ]:
backend = FilesystemBackend(root_dir=".", virtual_mode=False)

ki_agent = create_deep_agent(
    model=agent_llm,
    tools=[esql_query],
    skills=["skills"],
    backend=backend,
    system_prompt=(
        "You are a research assistant answering questions about a document corpus. "
        "You have NOT memorized the corpus. When a question depends on specific facts, "
        "names, dates, or events, use the query-ki skill to retrieve Knowledge "
        "Indicators before answering. Ground your answer strictly in what it returns, "
        "and cite the KI titles you used."
    ),
)

run_agent(ki_agent, QUESTION)

That's the full loop: documents went in, the workflow distilled each into a fact KI, the AI Index made them retrievable with one ES|QL query, and the agent answered from them. The KI answer is about equivalent, but reaches it in fewer tool calls and less time. The agent skips reading full documents and pulling hundreds of words of body text into context. That saving compounds across every query.

Exact tool-call counts, latency, and answers vary run to run, since the agent is non-deterministic. The precomputed-context pattern is the takeaway, not the specific numbers.

## Clean up

Remove the AI Index, the source index, and the workflow.

In [ ]:
client.indices.delete(index=AI_INDEX, ignore_unavailable=True)
print("Deleted index:", AI_INDEX)

client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
print("Deleted index:", INDEX_NAME)

try:
    kbn_request(
        "DELETE", f"/api/workflows/workflow/{workflow_id}", api_version=WF_API_VERSION
    )
    print("Deleted workflow:", workflow_id)
except Exception as e:
    print("Workflow delete skipped:", e)